In [101]:
import numpy as np
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
list(iris)

['data',
 'target',
 'frame',
 'target_names',
 'DESCR',
 'feature_names',
 'filename',
 'data_module']

In [102]:
list(iris.data)

['sepal length (cm)',
 'sepal width (cm)',
 'petal length (cm)',
 'petal width (cm)']

In [103]:
iris.data

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2
...,...,...,...,...
145,6.7,3.0,5.2,2.3
146,6.3,2.5,5.0,1.9
147,6.5,3.0,5.2,2.0
148,6.2,3.4,5.4,2.3


In [104]:
print(iris.data.shape)
print(iris.target.shape)

(150, 4)
(150,)


In [105]:
dataset = np.c_[iris.data, iris.target]
dataset = np.c_[np.ones(dataset.shape[0]), dataset]

In [106]:
test_ratio = 0.2
np.random.seed(42)
np.random.shuffle(dataset)
train, test = dataset[:int(len(dataset) * (1-test_ratio)), :], dataset[int(len(dataset) * (1-test_ratio)):, :]
X_train, y_train, X_test, y_test = train[:, :-1], train[:, -1], test[:, :-1], test[:, -1]
feature_number = X_test.shape[1]
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(120, 5) (120,) (30, 5) (30,)


In [107]:
class_number = len(np.unique(y_train))
parameter_matrix = np.ones((class_number, feature_number))
parameter_matrix.shape

(3, 5)

In [108]:
def sm_get_proba(X, parameter_matrix):
    scores = X @ parameter_matrix.T
    shifted_scores = scores - np.max(scores, axis=1, keepdims=True)
    top = np.exp(shifted_scores)
    down = np.sum(top, axis=1, keepdims=True)
    return top / down

def cross_entropy_gradient_vector(X, y, parameter_matrix):
    m = len(y)
    probs = sm_get_proba(X, parameter_matrix)
    
    correct_logprobs = -np.log(probs[np.arange(m), y] + 1e-9)
    loss = np.sum(correct_logprobs) / m
    
    Y_one_hot = np.zeros_like(probs)
    Y_one_hot[np.arange(m), y] = 1
    
    gradient  = (probs - Y_one_hot).T @ X/m
    
    return gradient, loss

In [109]:
def batch_gradient_descent(rate, epoches, X_train, y_train, parameter_matrix):
    for epoch in range(epoches):
        gradient, loss = cross_entropy_gradient_vector(X_train, y_train, parameter_matrix)
        parameter_matrix = parameter_matrix - rate * gradient
        if epoch % 1000 == 0:
            print(f"Loss: {loss:.4f}")
    return parameter_matrix
y_train = np.astype(y_train, int)
parameters = batch_gradient_descent(0.5, 5001, X_train, y_train, parameter_matrix)
parameters

Loss: 1.0986
Loss: 0.0692
Loss: 0.0644
Loss: 0.0616
Loss: 0.0596
Loss: 0.0582


array([[ 1.98318083,  3.05467936,  6.09460294, -5.53844432, -2.27929344],
       [ 6.02199784,  1.96215931,  0.99359   ,  1.15915592, -3.23971504],
       [-5.00517867, -2.01683867, -4.08819294,  7.3792884 ,  8.51900848]])

In [110]:
def predict(X, parameter_matrix):
    proba = sm_get_proba(X, parameter_matrix)
    return proba

predictions = predict(X_test, parameters)
predictions = np.argmax(predictions, axis=1)
accuracy = np.mean((predictions == y_test))
accuracy

np.float64(0.9666666666666667)

In [111]:
(predictions == y_test)

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True, False,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True])

In [112]:
predictions

array([1, 0, 1, 1, 0, 1, 2, 2, 0, 1, 2, 1, 0, 2, 0, 1, 2, 2, 1, 2, 1, 1,
       2, 2, 0, 1, 2, 0, 1, 2])

In [113]:
y_test

array([1., 0., 1., 1., 0., 1., 2., 2., 0., 1., 2., 2., 0., 2., 0., 1., 2.,
       2., 1., 2., 1., 1., 2., 2., 0., 1., 2., 0., 1., 2.])